# E3 — Low-light enhancement trước inference

E3 dùng checkpoint E1 đã train; không train lại model. CLAHE và Gamma được so sánh trên validation low-light subset, sau đó chỉ method được chọn mới được chạy trên test.

Chuẩn bị từ repository root:

```bash
uv sync --extra dev --extra enhancement
uv pip install git+https://github.com/THU-MIG/yolov10.git
uv run --with jupyterlab jupyter lab
```

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from pprint import pprint

candidate = Path.cwd().resolve()
for directory in (candidate, *candidate.parents):
    if (directory / 'pyproject.toml').is_file():
        PROJECT_ROOT = directory
        break
else:
    raise RuntimeError('Không tìm thấy project root (pyproject.toml).')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
E3_CONFIG = PROJECT_ROOT / 'configs' / 'E3_enhancement.yaml'
E1_CHECKPOINT = PROJECT_ROOT / 'experiments' / 'E1' / 'baseline_seed42' / 'weights' / 'best.pt'

## 1. Kiểm tra protocol, dataset và checkpoint

Cập nhật `E1_CHECKPOINT` nếu tên E1 run của bạn khác `baseline_seed42`.

In [ ]:
from helmet_yolov10.data.validation import validate_dataset
from helmet_yolov10.utils.config import load_config

config = load_config(E3_CONFIG)
assert config['experiment']['id'] == 'E3'
assert E1_CHECKPOINT.is_file(), f'Không tìm thấy checkpoint: {E1_CHECKPOINT}'
dataset_report = validate_dataset(PROJECT_ROOT / config['training']['data'], PROJECT_ROOT)
pprint(config['enhancement'])
pprint(dataset_report)

## 2. Xem trước CLAHE và Gamma

Đặt đường dẫn một ảnh validation thuộc low-light subset. Preview là kiểm tra chất lượng trực quan, không dùng test set để chọn tham số.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from helmet_yolov10.enhancement.pipeline import apply_enhancement

LOW_LIGHT_VAL_IMAGE = None  # Ví dụ: PROJECT_ROOT / 'data/processed/images/val/example.jpg'

if LOW_LIGHT_VAL_IMAGE is None:
    print('Đặt LOW_LIGHT_VAL_IMAGE thành một ảnh validation low-light để xem preview.')
else:
    image = cv2.imread(str(LOW_LIGHT_VAL_IMAGE), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(LOW_LIGHT_VAL_IMAGE)
    methods = config['enhancement']['candidates']
    previews = {'original': image, **{name: apply_enhancement(image, name, values) for name, values in methods.items()}}
    figure, axes = plt.subplots(1, len(previews), figsize=(5 * len(previews), 5))
    for axis, (name, bgr) in zip(axes, previews.items()):
        axis.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        axis.set_title(name)
        axis.axis('off')
    plt.tight_layout()

## 3. Tạo validation dataset đã enhancement

Cell này chỉ tạo ảnh mới cho split đang đánh giá; label được symlink từ dataset gốc. Train và test gốc không bị thay đổi. Mỗi method có thư mục riêng để backend YOLO đo mAP trên validation.

In [ ]:
import shutil
import yaml

IMAGE_SUFFIXES = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}

def stage_enhanced_split(method: str, split: str, stage_root: Path) -> Path:
    data_path = PROJECT_ROOT / config['training']['data']
    data = yaml.safe_load(data_path.read_text(encoding='utf-8'))
    source_root = Path(data['path'])
    if not source_root.is_absolute():
        source_root = PROJECT_ROOT / source_root
    source_images = source_root / data[split]
    destination_images = stage_root / 'images' / split
    destination_labels = stage_root / 'labels' / split
    if stage_root.exists():
        raise FileExistsError(f'Stage directory already exists: {stage_root}')
    destination_images.mkdir(parents=True)
    destination_labels.parent.mkdir(parents=True)
    os.symlink(source_root / 'labels' / split, destination_labels, target_is_directory=True)
    for source_image in source_images.rglob('*'):
        if source_image.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        relative = source_image.relative_to(source_images)
        destination_image = destination_images / relative
        destination_image.parent.mkdir(parents=True, exist_ok=True)
        image = cv2.imread(str(source_image), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'Cannot read image: {source_image}')
        enhanced = apply_enhancement(image, method, config['enhancement']['candidates'][method])
        if not cv2.imwrite(str(destination_image), enhanced):
            raise IOError(f'Cannot write image: {destination_image}')
    staged_data = dict(data)
    staged_data[split] = str(destination_images.resolve())
    staged_yaml = stage_root / 'data.yaml'
    staged_yaml.write_text(yaml.safe_dump(staged_data, sort_keys=False), encoding='utf-8')
    return staged_yaml

## 4. So sánh candidate trên validation

Bật cờ này sau khi kiểm tra preview. Nó chạy E1 checkpoint với từng candidate trên validation, ghi metric vào `experiments/E3/validation_candidates/`. Không chạy test ở bước này.

In [ ]:
import json

from helmet_yolov10.evaluation.evaluate import _as_serialisable_metrics
from helmet_yolov10.training.train import _load_backend

RUN_VALIDATION_COMPARISON = False
DEVICE = 0
VALIDATION_ROOT = PROJECT_ROOT / 'experiments' / 'E3' / 'validation_candidates'

if RUN_VALIDATION_COMPARISON:
    model = _load_backend()(str(E1_CHECKPOINT))
    evaluation = dict(config['evaluation'])
    evaluation.update({'device': DEVICE, 'split': 'val', 'exist_ok': True})
    for method in config['enhancement']['candidates']:
        method_root = VALIDATION_ROOT / method
        staged_data = stage_enhanced_split(method, 'val', method_root / 'dataset')
        result = model.val(data=str(staged_data), project=str(VALIDATION_ROOT), name=method, **evaluation)
        metrics = _as_serialisable_metrics(result)
        (method_root / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
        print(method, metrics)
else:
    print('Chưa chạy validation comparison. Đổi RUN_VALIDATION_COMPARISON thành True.')

## 5. Khóa method được chọn

Dựa trên metric validation đã lưu, đặt `SELECTED_METHOD` là `clahe` hoặc `gamma`. Selection file là artefact để giải trình rằng test set không được dùng chọn tham số.

In [ ]:
SELECTED_METHOD = None  # 'clahe' hoặc 'gamma' sau validation.

if SELECTED_METHOD is not None:
    if SELECTED_METHOD not in config['enhancement']['candidates']:
        raise ValueError(f'Method không hợp lệ: {SELECTED_METHOD}')
    selection_path = PROJECT_ROOT / 'experiments' / 'E3' / 'selection.yaml'
    selection_path.parent.mkdir(parents=True, exist_ok=True)
    selection_path.write_text(yaml.safe_dump({
        'selected_on_split': 'val',
        'selection_metric': config['enhancement']['selection_metric'],
        'method': SELECTED_METHOD,
        'parameters': config['enhancement']['candidates'][SELECTED_METHOD],
    }, sort_keys=False), encoding='utf-8')
    print(f'Đã khóa selection tại {selection_path}')
else:
    print('Chọn method sau validation; test chưa được chạy.')

## 6. Đánh giá test một lần sau khi khóa selection

Chỉ bật cell này khi `selection.yaml` đã được tạo ở bước 5. Không đổi method/tham số theo kết quả test.

In [ ]:
RUN_TEST_EVALUATION = False

if RUN_TEST_EVALUATION:
    selection_path = PROJECT_ROOT / 'experiments' / 'E3' / 'selection.yaml'
    if not selection_path.is_file():
        raise FileNotFoundError('Hãy khóa selection ở bước 5 trước khi chạy test.')
    selection = yaml.safe_load(selection_path.read_text(encoding='utf-8'))
    method = selection['method']
    test_root = PROJECT_ROOT / 'experiments' / 'E3' / 'test' / method
    staged_data = stage_enhanced_split(method, 'test', test_root / 'dataset')
    model = _load_backend()(str(E1_CHECKPOINT))
    evaluation = dict(config['evaluation'])
    evaluation.update({'device': DEVICE, 'split': 'test', 'exist_ok': True})
    result = model.val(data=str(staged_data), project=str(test_root), name='evaluation', **evaluation)
    metrics = _as_serialisable_metrics(result)
    (test_root / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print(metrics)
else:
    print('Test chưa chạy. Đổi RUN_TEST_EVALUATION thành True sau khi khóa selection.')